In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.linear_model import Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error , r2_score

from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("/content/gurgaon_properties_post_feature_selection_v2.csv")

In [ ]:
df.drop(columns=['pooja room','others','study room'],inplace=True)

In [ ]:
df['sector'] = df['sector'].str.lower().str.strip()
df['facing'] = df['facing'].str.lower().str.strip()
df['agePossession'] = df['agePossession'].str.lower().str.strip()
df['floor_category'] = df['floor_category'].str.lower().str.strip()

In [ ]:
df.head()

,sector,price,bedRoom,bathroom,balcony,facing,agePossession,built_up_area,servant room,store room,floor_category
0,sector 106,2.85,3,3,3,east,relatively new,1653.00,1,0,mid floor
1,sector 112,6.25,3,5,3,north,moderately old,2873.18,1,0,mid floor
2,sector 79,3.50,3,3,3,north,under construction,1603.00,1,0,high floor
3,sector 90,2.85,4,4,3+,north-west,moderately old,2725.00,1,0,mid floor
4,sector 112,8.40,4,6,3+,west,moderately old,3655.35,1,0,mid floor


In [ ]:
X = df.drop('price',axis=1)
y = df['price']

In [ ]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

# Ordinal Encoding

In [ ]:
columns_to_encode = ['sector','facing','agePossession','floor_category','balcony']

In [ ]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom','bathroom','built_up_area','servant room','store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ],
    remainder='passthrough'
)

In [ ]:
# Creating a Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
# K-fold cross validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed,cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.6687727692590768)

In [ ]:
scores.std()

np.float64(0.03228739972521589)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['sector', 'facing',
                                                   'agePossession',
                                                   'floor_category',
                                                   'balcony'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
y_pred = np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.8065981182782231

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
model_output

[['linear_reg', np.float64(0.6687727692590768), 0.8065981182782231],
 ['svr', np.float64(0.7128765245726475), 0.7107224876659138],
 ['ridge', np.float64(0.6687838004020338), 0.8065050643616133],
 ['LASSO', np.float64(0.05194139296591156), 1.3108829239214457],
 ['decision tree', np.float64(0.7539803788571917), 0.6364128780983086],
 ['random forest', np.float64(0.8562722322637386), 0.5060966558817451],
 ['extra trees', np.float64(0.8232472327981879), 0.5514953317107816],
 ['gradient boosting', np.float64(0.8487248785612396), 0.5346243639678611],
 ['adaboost', np.float64(0.7420700946695272), 0.6778540339424383],
 ['mlp', np.float64(0.7011170991433131), 0.757835336537251],
 ['xgboost', np.float64(0.8751062142606468), 0.48905200462715304]]

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.875106,0.489052
5,random forest,0.856272,0.506097
7,gradient boosting,0.848725,0.534624
6,extra trees,0.823247,0.551495
4,decision tree,0.753980,0.636413
8,adaboost,0.742070,0.677854
1,svr,0.712877,0.710722
9,mlp,0.701117,0.757835
2,ridge,0.668784,0.806505
0,linear_reg,0.668773,0.806598


# OneHotEncoding

In [ ]:
# Creating a column transformer for preprocessing
columns_to_encode = ['floor_category','balcony']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','facing'])
    ],
    remainder='passthrough'
)

In [ ]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [ ]:
scores.mean()

np.float64(0.8885217182350331)

In [ ]:
scores.std()

np.float64(0.019878508754100795)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['floor_category',
                                                   'balcony']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sector', 'agePossession',
                                                   'facing'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
y_pred = np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.4123708000482586

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
0,linear_reg,0.888522,0.412371
2,ridge,0.887884,0.425337
6,extra trees,0.886730,0.434895
10,xgboost,0.889323,0.444291
9,mlp,0.858940,0.485541
1,svr,0.872702,0.491832
5,random forest,0.858065,0.512612
7,gradient boosting,0.836351,0.568586
4,decision tree,0.800112,0.595228
8,adaboost,0.638386,0.873458


# OneHotEncoding With PCA

In [ ]:
# Creating a column transformer for preprocessing
columns_to_encode = ['facing','floor_category','balcony']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),['sector','agePossession'])
    ],
    remainder='passthrough'
)

In [ ]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [ ]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [ ]:
scores.mean()

np.float64(0.6813076082000563)

In [ ]:
scores.std()

np.float64(0.03227613130121313)

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.702669,0.723020
5,random forest,0.714919,0.735163
6,extra trees,0.726764,0.736341
7,gradient boosting,0.724188,0.745440
2,ridge,0.681369,0.788500
0,linear_reg,0.681308,0.788871
1,svr,0.671616,0.802363
9,mlp,0.676677,0.815592
8,adaboost,0.586605,0.939491
4,decision tree,0.443537,1.014044


# Target Encoder

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.9 MB/s eta 0:00:00


In [ ]:
import category_encoders as ce

columns_to_encode = ['facing','floor_category','balcony']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [ ]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.8194931893901674)

In [ ]:
scores.std()

np.float64(0.024037653926168343)

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.882106,0.440522
6,extra trees,0.884822,0.444166
10,xgboost,0.886760,0.477314
7,gradient boosting,0.874536,0.507134
4,decision tree,0.793843,0.553159
2,ridge,0.819522,0.577207
0,linear_reg,0.819493,0.577428
9,mlp,0.806022,0.594600
1,svr,0.808118,0.595335
8,adaboost,0.815056,0.624019


# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['sqrt', 'log2', None]
}

In [ ]:
columns_to_encode = ['facing','floor_category','balcony']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

In [ ]:
# search.fit(X, y_transformed)

In [ ]:
# final_pipe = search.best_estimator_

In [ ]:
# search.best_params_

In [ ]:
# search.best_score_

In [ ]:
# final_pipe.fit(X,y_transformed)

In [ ]:
# y_pred = final_pipe.predict(X_test)

In [ ]:
# y_pred = np.expm1(y_pred)

In [ ]:
# mean_absolute_error(np.expm1(y_test),y_pred)

# Exporting the model

In [ ]:
columns_to_encode = ['floor_category','balcony']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','facing'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500,random_state=42))
])

In [ ]:
pipeline1 = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

In [80]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [81]:
scores.mean()

np.float64(0.8597962336889513)

In [82]:
scores.std()

np.float64(0.015981951127432364)

In [83]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [84]:
pipeline.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['floor_category',
                                                   'balcony']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sector', 'agePossession',
                                                   'facing'])])),
                ('regressor',
                 RandomForestRegressor(n_estimators=500, random_state=42))])

In [ ]:
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

In [ ]:
r2_score(np.expm1(y_test),y_pred)

In [ ]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [89]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [90]:
X

,sector,bedRoom,bathroom,balcony,facing,agePossession,built_up_area,servant room,store room,floor_category
0,sector 106,3,3,3,east,relatively new,1653.00,1,0,mid floor
1,sector 112,3,5,3,north,moderately old,2873.18,1,0,mid floor
2,sector 79,3,3,3,north,under construction,1603.00,1,0,high floor
3,sector 90,4,4,3+,north-west,moderately old,2725.00,1,0,mid floor
4,sector 112,4,6,3+,west,moderately old,3655.35,1,0,mid floor
...,...,...,...,...,...,...,...,...,...,...
1523,sector 76,2,2,3,west,relatively new,834.00,0,0,high floor
1524,sector 69,3,3,2,south,new property,1470.00,0,0,mid floor
1525,sector 70,4,5,3+,east,moderately old,2150.00,1,0,mid floor
1526,sector 66,4,4,3+,west,moderately old,1711.00,1,0,mid floor


# Trying out the predictions

In [91]:
X.columns

Index(['sector', 'bedRoom', 'bathroom', 'balcony', 'facing', 'agePossession',
       'built_up_area', 'servant room', 'store room', 'floor_category'],
      dtype='object')

In [92]:
X.iloc[0].values

array(['sector 106', np.int64(3), np.int64(3), '3', 'east',
       'relatively new', np.float64(1653.0), np.int64(1), np.int64(0),
       'mid floor'], dtype=object)

In [93]:
y.iloc[0]

np.float64(2.85)

In [94]:
data = [['sector 49', 4, 3, '3+', 'east', 'new property', 1750, 0, 0,'low floor']]
columns = ['sector', 'bedRoom', 'bathroom', 'balcony',
       'facing', 'agePossession','built_up_area', 'servant room', 'store room',
      'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,sector,bedRoom,bathroom,balcony,facing,agePossession,built_up_area,servant room,store room,floor_category
0,sector 49,4,3,3+,east,new property,1750,0,0,low floor


In [95]:
np.expm1(pipeline.predict(one_df))

array([2.78899922])

In [96]:
X_train_pred = pipeline.predict(X_train)
X_test_pred = pipeline.predict(X_test)

print("Train R2:", r2_score(np.expm1(y_train), np.expm1(X_train_pred)))
print("Test R2:", r2_score(np.expm1(y_test), np.expm1(X_test_pred)))

Train R2: 0.9733368847512271
Test R2: 0.9691560257880061


In [97]:
import pandas as pd

importances = pipeline.named_steps['regressor'].feature_importances_
features = pipeline.named_steps['preprocessor'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': features, 'importance': importances})
imp_df.sort_values(by='importance', ascending=False).head(10)

,feature,importance
2,num__built_up_area,0.688250
50,cat1__sector_sector 54,0.030530
46,cat1__sector_sector 48,0.015510
55,cat1__sector_sector 59,0.015292
28,cat1__sector_sector 108,0.012998
40,cat1__sector_sector 37c,0.012380
87,cat1__sector_sector 92,0.012046
49,cat1__sector_sector 53,0.011061
61,cat1__sector_sector 65,0.010920
1,num__bathroom,0.009228


In [98]:
pipeline.named_steps['preprocessor']

ColumnTransformer(remainder='passthrough',
                  transformers=[('num', StandardScaler(),
                                 ['bedRoom', 'bathroom', 'built_up_area',
                                  'servant room', 'store room']),
                                ('cat',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['floor_category', 'balcony']),
                                ('cat1',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['sector', 'agePossession', 'facing'])])

In [99]:
import numpy as np
y_shuffled = np.random.permutation(y_train)

pipeline.fit(X_train, y_shuffled)
pipeline.score(X_test, y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


-0.0954936823851058

In [ ]:
X.dtypes

In [101]:
import sklearn
print(sklearn.__version__)

1.6.1
